# Обслуживание Iceberg-таблиц

Iceberg ничего не удаляет сам: каждый INSERT — новый снапшот и новые файлы,
каждый merge-on-read DELETE — ещё и delete-файлы. Без ухода таблица обрастает
мусором, а часть ридеров (привет, ClickHouse) вообще отказывается её читать.
Здесь — весь джентльменский набор процедур на демо-таблице.

Все процедуры — `CALL hive.system.<имя>(...)`, выполняются Spark'ом.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("maintenance").getOrCreate()

spark.sql("CREATE DATABASE IF NOT EXISTS hive.demo")
spark.sql("DROP TABLE IF EXISTS hive.demo.maint PURGE")
# merge-on-read: DELETE будет писать delete-файлы, а не переписывать данные
spark.sql("""CREATE TABLE hive.demo.maint (id BIGINT, v STRING) USING iceberg
             TBLPROPERTIES ('write.delete.mode'='merge-on-read')""")
for i, v in enumerate(["a", "b", "c", "d"], 1):
    spark.sql(f"INSERT INTO hive.demo.maint VALUES ({i}, '{v}')")
print("4 INSERT'а = 4 снапшота = 4 мелких файла")

## Компакция — rewrite_data_files

Каждый мелкий INSERT породил отдельный parquet. На проде такие файлы копятся
тысячами и убивают скорость чтения. Компакция склеивает их.

In [ ]:
print("файлов до: ", spark.sql("SELECT count(*) FROM hive.demo.maint.files").collect()[0][0])
spark.sql("CALL hive.system.rewrite_data_files(table => 'demo.maint', options => map('min-input-files','2'))").show()
print("файлов после:", spark.sql("SELECT count(*) FROM hive.demo.maint.files").collect()[0][0])

## Чистка истории — expire_snapshots

Снапшоты копятся при каждой записи. Старые держат на S3 уже не нужные файлы.
`older_than` в будущем + `retain_last => 1` — «снести всю историю, оставить текущее»
(в учебных целях; на проде держат дни/недели для time travel).

In [ ]:
print("снапшотов до: ", spark.sql("SELECT count(*) FROM hive.demo.maint.snapshots").collect()[0][0])
spark.sql("""CALL hive.system.expire_snapshots(table => 'demo.maint',
             older_than => TIMESTAMP '2030-01-01 00:00:00', retain_last => 1)""").show()
print("снапшотов после:", spark.sql("SELECT count(*) FROM hive.demo.maint.snapshots").collect()[0][0])

## Delete-файлы: создаём проблему…

DELETE в режиме merge-on-read не трогает данные — пишет рядом position-delete файл.
Дёшево при записи, но дорого при чтении, а ClickHouse `icebergS3()` на таком
падает с `positional and equality deletes are not supported`.

In [ ]:
spark.sql("DELETE FROM hive.demo.maint WHERE id = 2")
print("delete-файлов:", spark.sql("SELECT count(*) FROM hive.demo.maint.delete_files").collect()[0][0])

## …и лечим её

Проверенный порядок:
1. `rewrite_data_files(rewrite-all)` — вшивает удаления в новые data-файлы;
2. `rewrite_position_delete_files(rewrite-all)` — выкидывает повисшие (dangling) delete-файлы;
3. `expire_snapshots` — чтобы старые снапшоты не держали мусор на S3.

In [ ]:
spark.sql("CALL hive.system.rewrite_data_files(table => 'demo.maint', options => map('rewrite-all','true'))").show()
spark.sql("CALL hive.system.rewrite_position_delete_files(table => 'demo.maint', options => map('rewrite-all','true'))").show()
spark.sql("""CALL hive.system.expire_snapshots(table => 'demo.maint',
             older_than => TIMESTAMP '2030-01-01 00:00:00', retain_last => 1)""").show()
print("delete-файлов:", spark.sql("SELECT count(*) FROM hive.demo.maint.delete_files").collect()[0][0])
spark.sql("SELECT * FROM hive.demo.maint ORDER BY id").show()

## Сироты — remove_orphan_files

Файлы, на которые не ссылается ни один снапшот (обломки упавших джобов).
По умолчанию трогает только файлы старше 3 суток — и это правильно: агрессивный
`older_than` на живой таблице может снести файлы незакоммиченной записи.

```sql
CALL hive.system.remove_orphan_files(table => 'demo.maint')
```

## Прибраться

`PURGE` удаляет и данные в MinIO, не только запись в каталоге.

In [ ]:
spark.sql("DROP TABLE IF EXISTS hive.demo.maint PURGE")
print("демо-таблица удалена")

## Шпаргалка

| Симптом | Лекарство |
|---|---|
| тысячи мелких файлов | `rewrite_data_files` |
| таблица растёт, история не нужна | `expire_snapshots` |
| ClickHouse не читает после DELETE | `rewrite_data_files(rewrite-all)` + `rewrite_position_delete_files(rewrite-all)` |
| мусор от упавших джобов | `remove_orphan_files` (с дефолтным older_than!) |